### PyTorch Tensors

In [1]:
import torch

In [2]:
X = torch.tensor([[1.0, 4.0, 7.0], [2.0, 3.0, 6.0]])
X

tensor([[1., 4., 7.],
        [2., 3., 6.]])

In [3]:
X.shape

torch.Size([2, 3])

In [4]:
X.dtype

torch.float32

In [5]:
X[0, 1]

tensor(4.)

In [6]:
X[:, 1]

tensor([4., 3.])

In [7]:
10 * (X + 1.0)

tensor([[20., 50., 80.],
        [30., 40., 70.]])

In [8]:
X.exp()

tensor([[   2.7183,   54.5982, 1096.6332],
        [   7.3891,   20.0855,  403.4288]])

In [9]:
X.mean()

tensor(3.8333)

In [10]:
X.max(dim=0)

torch.return_types.max(
values=tensor([2., 4., 7.]),
indices=tensor([1, 0, 0]))

In [11]:
X @ X.T

tensor([[66., 56.],
        [56., 49.]])

In [12]:
import numpy as np

X.numpy()

array([[1., 4., 7.],
       [2., 3., 6.]], dtype=float32)

In [13]:
torch.tensor(np.array([[1., 4., 7.], [2., 3., 6.]]))

tensor([[1., 4., 7.],
        [2., 3., 6.]], dtype=torch.float64)

In [14]:
torch.tensor(np.array([[1., 4., 7.], [2., 3., 6.]]), dtype=torch.float32)

tensor([[1., 4., 7.],
        [2., 3., 6.]])

In [15]:
torch.FloatTensor(np.array([[1., 4., 7.], [2., 3., 6.]]))

tensor([[1., 4., 7.],
        [2., 3., 6.]])

In [16]:
torch.from_numpy(np.array([[1., 4., 7.], [2., 3., 6.]]))

tensor([[1., 4., 7.],
        [2., 3., 6.]], dtype=torch.float64)

In [17]:
X[:, 1] = -99
X


tensor([[  1., -99.,   7.],
        [  2., -99.,   6.]])

In [18]:
X.relu_()
X

tensor([[1., 0., 7.],
        [2., 0., 6.]])

### Hardware Acceleration

In [19]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.mps.is_available():
    device = "mps"
else:
    device = "cpu"
device

'mps'

In [20]:
M = torch.tensor([[1., 2., 3.], [4., 5., 6.]])
M = M.to(device)

In [21]:
M.device

device(type='mps', index=0)

In [22]:
M = torch.tensor([[1., 2., 3.], [4., 5., 6.]], device=device)

In [23]:
R = M @ M.T
R

tensor([[14., 32.],
        [32., 77.]], device='mps:0')

In [24]:
M = torch.rand(1000, 1000)
%timeit M @ M.T

1.06 ms ± 21.7 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [25]:
M = torch.rand(1000, 1000, device="mps")
%timeit M @ M.T

242 μs ± 66.3 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


The `timeit` average is about 5 times lower with the GPU, but the overall (clock) time is more than double. Perhaps because the bottleneck is moving data in and out of the GPU.

### Autograd

In [26]:
x = torch.tensor(5.0, requires_grad=True)
f = x ** 2
f

tensor(25., grad_fn=<PowBackward0>)

In [27]:
f.backward()
x.grad

tensor(10.)

In [28]:
# gradient descent step (disabling gradient tracking)
learning_rate = 0.1
with torch.no_grad():
    x -= learning_rate * x.grad
x

tensor(4., requires_grad=True)

In [29]:
x_detached = x.detach()
x_detached -= learning_rate * x.grad
x_detached

tensor(3.)

In [30]:
# modifying x_detached also modifies x
x

tensor(3., requires_grad=True)

In [31]:
x.grad.zero_()

tensor(0.)

In [32]:
# Training loop
learning_rate = 0.1
x = torch.tensor(5.0, requires_grad=True)
for iteration in range(100):
    f = x ** 2  # forward pass
    f.backward()  # backward pass, computes the gradients
    with torch.no_grad():
        x -= learning_rate * x.grad  # gradient descent step
    
    x.grad.zero_()

x

tensor(1.0185e-09, requires_grad=True)

In [33]:
t = torch.tensor(2.0, requires_grad=True)
z = t.exp()  # this is an intermediate result
z +=1  # in-place operation
# z.backward()  # RuntimeError

In [34]:
t = torch.tensor(2.0, requires_grad=True)
z = t.exp()  # this is an intermediate result
z = z + 1  # remove in-place operation
z.backward()

### Linear Regression

In [35]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

housing = fetch_california_housing()
X_train_full, X_test, y_train_full, y_test = train_test_split(
    housing.data, housing.target, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, random_state=42)

In [36]:
X_train = torch.FloatTensor(X_train)
X_valid = torch.FloatTensor(X_valid)
X_test = torch.FloatTensor(X_test)
means = X_train.mean(dim=0, keepdims=True)
stds = X_train.std(dim=0, keepdims=True)
X_train = (X_train - means) / stds
X_valid = (X_valid - means) / stds
X_test = (X_test - means) / stds

In [37]:
y_train = torch.FloatTensor(y_train).reshape(-1, 1)
y_valid = torch.FloatTensor(y_valid).reshape(-1, 1)
y_test = torch.FloatTensor(y_test).reshape(-1, 1)

In [38]:
torch.manual_seed(42)
n_features = X_train.shape[1]  # 8 input features
w = torch.randn((n_features, 1), requires_grad=True)
b = torch.tensor(0., requires_grad=True)

In [39]:
# Batch Gradient Descent using autograd
learning_rate = 0.4
n_epochs = 20
for epoch in range(n_epochs):
    y_pred = X_train @ w + b
    loss = ((y_pred - y_train) ** 2).mean()
    loss.backward()
    with torch.no_grad():
        b -= learning_rate * b.grad
        w -= learning_rate * w.grad
        b.grad.zero_()
        w.grad.zero_()
    print(f"Epoch {epoch + 1} / {n_epochs}, Loss: {loss.item()}")

Epoch 1 / 20, Loss: 16.158458709716797
Epoch 2 / 20, Loss: 4.87937593460083
Epoch 3 / 20, Loss: 2.255227565765381
Epoch 4 / 20, Loss: 1.330764651298523
Epoch 5 / 20, Loss: 0.9680710434913635
Epoch 6 / 20, Loss: 0.8142688870429993
Epoch 7 / 20, Loss: 0.7417054176330566
Epoch 8 / 20, Loss: 0.702070951461792
Epoch 9 / 20, Loss: 0.6765925288200378
Epoch 10 / 20, Loss: 0.65779709815979
Epoch 11 / 20, Loss: 0.6426157355308533
Epoch 12 / 20, Loss: 0.6297228336334229
Epoch 13 / 20, Loss: 0.6184946298599243
Epoch 14 / 20, Loss: 0.6085972189903259
Epoch 15 / 20, Loss: 0.5998220443725586
Epoch 16 / 20, Loss: 0.5920190215110779
Epoch 17 / 20, Loss: 0.5850694179534912
Epoch 18 / 20, Loss: 0.5788736343383789
Epoch 19 / 20, Loss: 0.5733456015586853
Epoch 20 / 20, Loss: 0.5684102773666382


In [40]:
X_new = X_test[:3]
with torch.no_grad():
    y_pred = X_new @ w + b
y_pred

tensor([[0.8916],
        [1.6480],
        [2.6577]])

### Linear Regression using High-Level API

In [41]:
import torch.nn as nn   # Convention
torch.manual_seed(42)
model = nn.Linear(in_features=n_features, out_features=1)

In [42]:
model.bias

Parameter containing:
tensor([0.3117], requires_grad=True)

In [43]:
model.weight

Parameter containing:
tensor([[ 0.2703,  0.2935, -0.0828,  0.3248, -0.0775,  0.0713, -0.1721,  0.2076]],
       requires_grad=True)

In [44]:
for param in model.parameters():
    print(param)

Parameter containing:
tensor([[ 0.2703,  0.2935, -0.0828,  0.3248, -0.0775,  0.0713, -0.1721,  0.2076]],
       requires_grad=True)
Parameter containing:
tensor([0.3117], requires_grad=True)


In [45]:
model(X_train[:2])

tensor([[-0.4718],
        [ 0.1131]], grad_fn=<AddmmBackward0>)

In [46]:
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()

In [47]:
def train_bgd(model, optimizer, criterion, X_train, y_train, n_epochs):
    for epoch in range(n_epochs):
        y_pred = model(X_train)
        loss = criterion(y_pred, y_train)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        print(f"Epoch {epoch + 1} / {n_epochs}, Loss: {loss.item()}")

In [48]:
train_bgd(model, optimizer, mse, X_train, y_train, n_epochs)

Epoch 1 / 20, Loss: 4.3378496170043945
Epoch 2 / 20, Loss: 0.7802932262420654
Epoch 3 / 20, Loss: 0.6253839731216431
Epoch 4 / 20, Loss: 0.6060433983802795
Epoch 5 / 20, Loss: 0.595629870891571
Epoch 6 / 20, Loss: 0.5873565673828125
Epoch 7 / 20, Loss: 0.5802989602088928
Epoch 8 / 20, Loss: 0.5741382241249084
Epoch 9 / 20, Loss: 0.5687100291252136
Epoch 10 / 20, Loss: 0.5639079213142395
Epoch 11 / 20, Loss: 0.5596510767936707
Epoch 12 / 20, Loss: 0.5558737516403198
Epoch 13 / 20, Loss: 0.5525193810462952
Epoch 14 / 20, Loss: 0.5495391488075256
Epoch 15 / 20, Loss: 0.5468899011611938
Epoch 16 / 20, Loss: 0.544533908367157
Epoch 17 / 20, Loss: 0.5424376726150513
Epoch 18 / 20, Loss: 0.5405715703964233
Epoch 19 / 20, Loss: 0.5389097332954407
Epoch 20 / 20, Loss: 0.5374287962913513


In [49]:
X_new = X_test[:3]
with torch.no_grad():
    y_pred = model(X_new)

y_pred

tensor([[0.8061],
        [1.7116],
        [2.6973]])

### Regression LMP

In [50]:
torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(n_features, 50),
    nn.ReLU(),
    nn.Linear(50, 40),
    nn.ReLU(),
    nn.Linear(40, 1)
)

In [51]:
learning_rate = 0.1
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()
train_bgd(model, optimizer, mse, X_train, y_train, n_epochs)

Epoch 1 / 20, Loss: 5.045480251312256
Epoch 2 / 20, Loss: 2.0523128509521484
Epoch 3 / 20, Loss: 1.00398850440979
Epoch 4 / 20, Loss: 0.8570139408111572
Epoch 5 / 20, Loss: 0.7740675210952759
Epoch 6 / 20, Loss: 0.7225848436355591
Epoch 7 / 20, Loss: 0.6893726587295532
Epoch 8 / 20, Loss: 0.6669033169746399
Epoch 9 / 20, Loss: 0.6507739424705505
Epoch 10 / 20, Loss: 0.6383934020996094
Epoch 11 / 20, Loss: 0.6281994581222534
Epoch 12 / 20, Loss: 0.6193398833274841
Epoch 13 / 20, Loss: 0.6113173365592957
Epoch 14 / 20, Loss: 0.6038705110549927
Epoch 15 / 20, Loss: 0.5968307852745056
Epoch 16 / 20, Loss: 0.5901119112968445
Epoch 17 / 20, Loss: 0.583646833896637
Epoch 18 / 20, Loss: 0.5774063467979431
Epoch 19 / 20, Loss: 0.5713555216789246
Epoch 20 / 20, Loss: 0.565444827079773


### Mini-Batch GD

In [92]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [93]:
torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(n_features, 50),
    nn.ReLU(),
    nn.Linear(50, 40),
    nn.ReLU(),
    nn.Linear(40, 1)
)
# model.to(device)

In [94]:
learning_rate = 0.02
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()

In [95]:
def train(model, optimizer, criterion, train_loader, n_epochs):
    model.train()
    for epoch in range(n_epochs):
        total_loss = 0.
        for X_batch, y_batch in train_loader:
            # X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
        mean_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch + 1} / {n_epochs}, Loss: {mean_loss:.4f}")

In [96]:
train(model, optimizer, mse, train_loader, n_epochs)

Epoch 1 / 20, Loss: 0.5900
Epoch 2 / 20, Loss: 0.4046
Epoch 3 / 20, Loss: 0.3801
Epoch 4 / 20, Loss: 0.3629
Epoch 5 / 20, Loss: 0.3529
Epoch 6 / 20, Loss: 0.3520
Epoch 7 / 20, Loss: 0.3408
Epoch 8 / 20, Loss: 0.3427
Epoch 9 / 20, Loss: 0.3406
Epoch 10 / 20, Loss: 0.3378
Epoch 11 / 20, Loss: 0.3304
Epoch 12 / 20, Loss: 0.3267
Epoch 13 / 20, Loss: 0.3244
Epoch 14 / 20, Loss: 0.3221
Epoch 15 / 20, Loss: 0.3186
Epoch 16 / 20, Loss: 0.3149
Epoch 17 / 20, Loss: 0.3123
Epoch 18 / 20, Loss: 0.3111
Epoch 19 / 20, Loss: 0.3088
Epoch 20 / 20, Loss: 0.3072


### Evaluation

In [97]:
def evaluate(model, data_loader, metric_fn, aggregate_fn = torch.mean):
    model.eval()
    metrics = []
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            # X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric = metric_fn(y_pred, y_batch)
            metrics.append(metric)
    return aggregate_fn(torch.stack(metrics))

In [98]:
valid_dataset = TensorDataset(X_valid, y_valid)
valid_loader = DataLoader(valid_dataset, batch_size=32)
valid_mse = evaluate(model, valid_loader, mse)
valid_mse

tensor(0.4080)

In [99]:
def rmse(y_pred, y_true):
    return ((y_pred - y_true) ** 2).mean().sqrt()

evaluate(model, valid_loader, rmse)

tensor(0.5668)

In [100]:
valid_mse.sqrt()

tensor(0.6388)

In [101]:
evaluate(model, valid_loader, mse, 
         aggregate_fn=lambda metrics: torch.sqrt(torch.mean(metrics)))

tensor(0.6388)

In [102]:
import torchmetrics

def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            # X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()

In [103]:
rmse = torchmetrics.MeanSquaredError(squared=False)
evaluate_tm(model, valid_loader, rmse)

tensor(0.6388)

In [108]:
def train(model, optimizer, criterion, train_loader, valid_loader, eval_metric, n_epochs):
    for epoch in range(n_epochs):
        model.train()
        total_loss = 0.
        for X_batch, y_batch in train_loader:
            # X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
        mean_loss = total_loss / len(train_loader)
        train_rmse = evaluate_tm(model, train_loader, eval_metric)
        valid_rmse = evaluate_tm(model, valid_loader, eval_metric)
        print(f"Epoch {epoch + 1} / {n_epochs}, Loss: {mean_loss:.4f}")
        print(f"Train rmse: {train_rmse.item():.3f}, Valid rmse: {valid_rmse.item():.3f}")

In [109]:
train(model, optimizer, mse, train_loader, valid_loader, rmse, n_epochs)

Epoch 1 / 20, Loss: 0.2778
Train rmse: 0.528, Valid rmse: 0.540
Epoch 2 / 20, Loss: 0.2764
Train rmse: 0.521, Valid rmse: 0.522
Epoch 3 / 20, Loss: 0.2766
Train rmse: 0.527, Valid rmse: 0.607
Epoch 4 / 20, Loss: 0.2762
Train rmse: 0.517, Valid rmse: 0.535
Epoch 5 / 20, Loss: 0.2744
Train rmse: 0.515, Valid rmse: 0.532
Epoch 6 / 20, Loss: 0.2736
Train rmse: 0.523, Valid rmse: 0.546
Epoch 7 / 20, Loss: 0.2758
Train rmse: 0.523, Valid rmse: 0.607
Epoch 8 / 20, Loss: 0.2712
Train rmse: 0.511, Valid rmse: 0.556
Epoch 9 / 20, Loss: 0.2713
Train rmse: 0.534, Valid rmse: 0.619
Epoch 10 / 20, Loss: 0.2710
Train rmse: 0.516, Valid rmse: 0.536
Epoch 11 / 20, Loss: 0.2708
Train rmse: 0.510, Valid rmse: 0.614
Epoch 12 / 20, Loss: 0.2715
Train rmse: 0.540, Valid rmse: 0.555
Epoch 13 / 20, Loss: 0.2706
Train rmse: 0.511, Valid rmse: 0.561
Epoch 14 / 20, Loss: 0.2675
Train rmse: 0.511, Valid rmse: 0.548
Epoch 15 / 20, Loss: 0.2668
Train rmse: 0.520, Valid rmse: 0.570
Epoch 16 / 20, Loss: 0.2681
Train 

### Nonsequential Models

In [111]:
class WideAndDeep(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.deep_stack = nn.Sequential(
            nn.Linear(n_features, 50), nn.ReLU(),
            nn.Linear(50, 40), nn.ReLU(),
        )
        self.output_layer = nn.Linear(40 + n_features, 1)  # 40 + n_features due to concat

    def forward(self, X):
        deep_output = self.deep_stack(X)
        wide_and_deep = torch.concat([X, deep_output], dim=1)
        return self.output_layer(wide_and_deep)


In [112]:
torch.manual_seed(42)
model = WideAndDeep(n_features)
learning_rate = 0.002
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()
train(model, optimizer, mse, train_loader, valid_loader, rmse, n_epochs)

Epoch 1 / 20, Loss: 1.4093
Train rmse: 0.797, Valid rmse: 0.879
Epoch 2 / 20, Loss: 0.6084
Train rmse: 0.761, Valid rmse: 0.793
Epoch 3 / 20, Loss: 0.5648
Train rmse: 0.739, Valid rmse: 0.873
Epoch 4 / 20, Loss: 0.5270
Train rmse: 0.716, Valid rmse: 0.795
Epoch 5 / 20, Loss: 0.5048
Train rmse: 0.700, Valid rmse: 0.705
Epoch 6 / 20, Loss: 0.4810
Train rmse: 0.687, Valid rmse: 0.699
Epoch 7 / 20, Loss: 0.4642
Train rmse: 0.676, Valid rmse: 0.688
Epoch 8 / 20, Loss: 0.4501
Train rmse: 0.666, Valid rmse: 0.643
Epoch 9 / 20, Loss: 0.4396
Train rmse: 0.659, Valid rmse: 0.664
Epoch 10 / 20, Loss: 0.4301
Train rmse: 0.652, Valid rmse: 0.651
Epoch 11 / 20, Loss: 0.4222
Train rmse: 0.647, Valid rmse: 0.645
Epoch 12 / 20, Loss: 0.4157
Train rmse: 0.642, Valid rmse: 0.645
Epoch 13 / 20, Loss: 0.4105
Train rmse: 0.638, Valid rmse: 0.620
Epoch 14 / 20, Loss: 0.4044
Train rmse: 0.633, Valid rmse: 0.735
Epoch 15 / 20, Loss: 0.4007
Train rmse: 0.629, Valid rmse: 0.623
Epoch 16 / 20, Loss: 0.3948
Train 

In [ ]:
class WideAndDeepV2(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.deep_stack = nn.Sequential(
            nn.Linear(n_features - 2, 50), nn.ReLU(),
            nn.Linear(50, 40), nn.ReLU(),
        )
        self.output_layer = nn.Linear(40 + n_features - 3, 1)

    def forward(self, X):
        X_wide = X[:, :5]
        X_deep = X[:, 2:]
        deep_output = self.deep_stack(X_deep)
        wide_and_deep = torch.concat([X_wide, deep_output], dim=1)
        return self.output_layer(wide_and_deep)

In [117]:
torch.manual_seed(42)
model = WideAndDeepV2(n_features)
learning_rate = 0.002
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()
train(model, optimizer, mse, train_loader, valid_loader, rmse, n_epochs)

Epoch 1 / 20, Loss: 1.4730
Train rmse: 0.803, Valid rmse: 0.988
Epoch 2 / 20, Loss: 0.5910
Train rmse: 0.744, Valid rmse: 0.756
Epoch 3 / 20, Loss: 0.5329
Train rmse: 0.716, Valid rmse: 0.706
Epoch 4 / 20, Loss: 0.4996
Train rmse: 0.697, Valid rmse: 0.685
Epoch 5 / 20, Loss: 0.4776
Train rmse: 0.684, Valid rmse: 0.671
Epoch 6 / 20, Loss: 0.4630
Train rmse: 0.676, Valid rmse: 0.667
Epoch 7 / 20, Loss: 0.4532
Train rmse: 0.669, Valid rmse: 0.663
Epoch 8 / 20, Loss: 0.4461
Train rmse: 0.665, Valid rmse: 0.662
Epoch 9 / 20, Loss: 0.4405
Train rmse: 0.662, Valid rmse: 0.669
Epoch 10 / 20, Loss: 0.4358
Train rmse: 0.658, Valid rmse: 0.667
Epoch 11 / 20, Loss: 0.4317
Train rmse: 0.656, Valid rmse: 0.671
Epoch 12 / 20, Loss: 0.4293
Train rmse: 0.653, Valid rmse: 0.675
Epoch 13 / 20, Loss: 0.4248
Train rmse: 0.651, Valid rmse: 0.675
Epoch 14 / 20, Loss: 0.4231
Train rmse: 0.648, Valid rmse: 0.676
Epoch 15 / 20, Loss: 0.4201
Train rmse: 0.647, Valid rmse: 0.671
Epoch 16 / 20, Loss: 0.4181
Train 

### Multiple Inputs

In [118]:
class WideAndDeepV3(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.deep_stack = nn.Sequential(
            nn.Linear(n_features - 2, 50), nn.ReLU(),
            nn.Linear(50, 40), nn.ReLU(),
        )
        self.output_layer = nn.Linear(40 + n_features - 3, 1)

    def forward(self, X_wide, X_deep):
        deep_output = self.deep_stack(X_deep)
        wide_and_deep = torch.concat([X_wide, deep_output], dim=1)
        return self.output_layer(wide_and_deep)

In [126]:
train_data_wd = TensorDataset(X_train[:, :5], X_train[:, 2:], y_train)
train_loader_wd = DataLoader(train_data_wd, batch_size=32, shuffle=True)
valid_data_wd = TensorDataset(X_valid[:, :5], X_valid[:, 2:], y_valid)
valid_loader_wd = DataLoader(valid_data_wd, batch_size=32, shuffle=True)
test_data_wd = TensorDataset(X_test[:, :5], X_test[:, 2:], y_test)
test_loader_wd = DataLoader(test_data_wd, batch_size=32, shuffle=True)

In [128]:
def evaluate_multiple_inputs(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        # for X_batch_wide, X_batch_deep, y_batch in data_loader:
        for *X_batch_inputs, y_batch in data_loader:
            # y_pred = model(X_batch_wide, X_batch_deep)
            y_pred = model(*X_batch_inputs)
            metric.update(y_pred, y_batch)
    return metric.compute()

def train_multiple_inputs(model, optimizer, criterion, train_loader, valid_loader, eval_metric, n_epochs):
    for epoch in range(n_epochs):
        model.train()
        total_loss = 0.
        # for X_batch_wide, X_batch_deep, y_batch in train_loader:
        for *X_batch_inputs, y_batch in train_loader:
            y_pred = model(*X_batch_inputs)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
        mean_loss = total_loss / len(train_loader)
        train_rmse = evaluate_multiple_inputs(model, train_loader, eval_metric)
        valid_rmse = evaluate_multiple_inputs(model, valid_loader, eval_metric)
        print(f"Epoch {epoch + 1} / {n_epochs}, Loss: {mean_loss:.4f}")
        print(f"Train rmse: {train_rmse.item():.3f}, Valid rmse: {valid_rmse.item():.3f}")

In [129]:
torch.manual_seed(42)
model = WideAndDeepV3(n_features)
learning_rate = 0.002
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()
train_multiple_inputs(model, optimizer, mse, train_loader_wd, valid_loader_wd, rmse, n_epochs)

Epoch 1 / 20, Loss: 1.4730
Train rmse: 0.803, Valid rmse: 0.988
Epoch 2 / 20, Loss: 0.5921
Train rmse: 0.744, Valid rmse: 0.759
Epoch 3 / 20, Loss: 0.5324
Train rmse: 0.716, Valid rmse: 0.708
Epoch 4 / 20, Loss: 0.4996
Train rmse: 0.698, Valid rmse: 0.686
Epoch 5 / 20, Loss: 0.4778
Train rmse: 0.685, Valid rmse: 0.672
Epoch 6 / 20, Loss: 0.4632
Train rmse: 0.676, Valid rmse: 0.665
Epoch 7 / 20, Loss: 0.4532
Train rmse: 0.669, Valid rmse: 0.663
Epoch 8 / 20, Loss: 0.4454
Train rmse: 0.665, Valid rmse: 0.665
Epoch 9 / 20, Loss: 0.4407
Train rmse: 0.662, Valid rmse: 0.665
Epoch 10 / 20, Loss: 0.4364
Train rmse: 0.659, Valid rmse: 0.667
Epoch 11 / 20, Loss: 0.4311
Train rmse: 0.656, Valid rmse: 0.670
Epoch 12 / 20, Loss: 0.4287
Train rmse: 0.654, Valid rmse: 0.670
Epoch 13 / 20, Loss: 0.4261
Train rmse: 0.651, Valid rmse: 0.671
Epoch 14 / 20, Loss: 0.4232
Train rmse: 0.649, Valid rmse: 0.674
Epoch 15 / 20, Loss: 0.4207
Train rmse: 0.649, Valid rmse: 0.673
Epoch 16 / 20, Loss: 0.4186
Train 

### Multiple Outputs

In [130]:
class WideAndDeepV4(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.deep_stack = nn.Sequential(
            nn.Linear(n_features - 2, 50), nn.ReLU(),
            nn.Linear(50, 40), nn.ReLU(),
        )
        self.output_layer = nn.Linear(40 + n_features - 3, 1)
        self.aux_output_layer = nn.Linear(40, 1)

    def forward(self, X_wide, X_deep):
        deep_output = self.deep_stack(X_deep)
        wide_and_deep = torch.concat([X_wide, deep_output], dim=1)
        main_output = self.output_layer(wide_and_deep)
        aux_output = self.aux_output_layer(deep_output)
        return main_output, aux_output

In [132]:
def evaluate_v4(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for *X_batch_inputs, y_batch in data_loader:
            y_pred, _ = model(*X_batch_inputs)  # ignore auxiliary output
            metric.update(y_pred, y_batch)
    return metric.compute()

def train_v4(model, optimizer, criterion, train_loader, valid_loader, eval_metric, n_epochs):
    for epoch in range(n_epochs):
        model.train()
        total_loss = 0.
        for *X_batch_inputs, y_batch in train_loader:
            y_pred, y_pred_aux = model(*X_batch_inputs)
            main_loss = criterion(y_pred, y_batch)
            aux_loss = criterion(y_pred_aux, y_batch)
            loss = 0.8 * main_loss + 0.2 * aux_loss
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
        mean_loss = total_loss / len(train_loader)
        train_rmse = evaluate_v4(model, train_loader, eval_metric)
        valid_rmse = evaluate_v4(model, valid_loader, eval_metric)
        print(f"Epoch {epoch + 1} / {n_epochs}, Loss: {mean_loss:.4f}")
        print(f"Train rmse: {train_rmse.item():.3f}, Valid rmse: {valid_rmse.item():.3f}")

In [133]:
torch.manual_seed(42)
model = WideAndDeepV4(n_features)
learning_rate = 0.002
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()
train_v4(model, optimizer, mse, train_loader_wd, valid_loader_wd, rmse, n_epochs)

Epoch 1 / 20, Loss: 1.9046
Train rmse: 0.836, Valid rmse: 1.250
Epoch 2 / 20, Loss: 0.7666
Train rmse: 0.764, Valid rmse: 0.837
Epoch 3 / 20, Loss: 0.6790
Train rmse: 0.733, Valid rmse: 0.726
Epoch 4 / 20, Loss: 0.6339
Train rmse: 0.712, Valid rmse: 0.694
Epoch 5 / 20, Loss: 0.6009
Train rmse: 0.696, Valid rmse: 0.677
Epoch 6 / 20, Loss: 0.5756
Train rmse: 0.685, Valid rmse: 0.667
Epoch 7 / 20, Loss: 0.5555
Train rmse: 0.677, Valid rmse: 0.659
Epoch 8 / 20, Loss: 0.5399
Train rmse: 0.671, Valid rmse: 0.658
Epoch 9 / 20, Loss: 0.5267
Train rmse: 0.667, Valid rmse: 0.658
Epoch 10 / 20, Loss: 0.5161
Train rmse: 0.664, Valid rmse: 0.660
Epoch 11 / 20, Loss: 0.5065
Train rmse: 0.661, Valid rmse: 0.667
Epoch 12 / 20, Loss: 0.4979
Train rmse: 0.658, Valid rmse: 0.670
Epoch 13 / 20, Loss: 0.4902
Train rmse: 0.655, Valid rmse: 0.675
Epoch 14 / 20, Loss: 0.4837
Train rmse: 0.653, Valid rmse: 0.680
Epoch 15 / 20, Loss: 0.4773
Train rmse: 0.650, Valid rmse: 0.683
Epoch 16 / 20, Loss: 0.4716
Train 